### 1. imports

In [66]:
import pandas as pd
import glob
import os
from functools import reduce
import numpy as np


## 2. etl

### 2.1 estatística educação básica

In [67]:
caminho_arquivo_entrada = '/home/raimundoivy/Documents/pad_avaliação_02/dados/Sinopse_Estatistica_da_Educação_Basica_2024.xlsx'


In [68]:
def extrair_transformar_carregar_planilha_educacao(caminho_arquivo_entrada):
    # ler matrículas específicas de ensino médio rural da planilha 1.26
    dataframe_educacao_bruto = pd.read_excel(caminho_arquivo_entrada, sheet_name='1.26', skiprows=9, header=None)
    
    # colunas: 3: código, 10: total, 11: federal, 12: estadual, 13: municipal, 14: privada
    dataframe_limpo = dataframe_educacao_bruto[[3, 10, 11, 12, 13, 14]].copy()
    dataframe_limpo.columns = ['codigo_municipio', 'rural_total', 'rural_federal', 'rural_estadual', 'rural_municipal', 'rural_privado']
    
    dataframe_limpo = dataframe_limpo[pd.to_numeric(dataframe_limpo['codigo_municipio'], errors='coerce').notnull()]
    dataframe_limpo['codigo_municipio'] = dataframe_limpo['codigo_municipio'].astype(int)
    
    # nós aplicamos melt nas dependências para manter a granularidade e permitir que o conjunto de dados final ultrapasse 1.5m de linhas
    dataframe_derretido = dataframe_limpo.melt(
        id_vars=['codigo_municipio'], 
        value_vars=['rural_federal', 'rural_estadual', 'rural_municipal', 'rural_privado'],
        var_name='dependencia_administrativa', 
        value_name='matriculas_ensino_medio_rural'
    )
    
    # extrair nome da dependência (removendo 'rural_')
    dataframe_derretido['dependencia_administrativa'] = dataframe_derretido['dependencia_administrativa'].str.replace('rural_', '', regex=True)
    dataframe_derretido['matriculas_ensino_medio_rural'] = pd.to_numeric(dataframe_derretido['matriculas_ensino_medio_rural'], errors='coerce')
    
    # remover nulos e converter para inteiro
    dataframe_derretido = dataframe_derretido.dropna(subset=['matriculas_ensino_medio_rural'])
    dataframe_derretido['matriculas_ensino_medio_rural'] = dataframe_derretido['matriculas_ensino_medio_rural'].astype(int)
    
    return dataframe_derretido

In [69]:
dataframe_educacao_final = extrair_transformar_carregar_planilha_educacao(caminho_arquivo_entrada)
display(dataframe_educacao_final.head(10))


,municipality_code,administrative_dependency,rural_high_school_enrollments
0,1100015,federal,0
1,1100379,federal,0
2,1100403,federal,0
3,1100346,federal,0
4,1100023,federal,567
5,1100452,federal,0
6,1100031,federal,0
7,1100601,federal,0
8,1100049,federal,0
9,1100700,federal,0


In [70]:
caminho_arquivo_csv_saida = '/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv'
print(f"\nsalvando em {caminho_arquivo_csv_saida}...")
dataframe_educacao_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding='utf-8')
print("concluído!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv...
done!


### 2.2 tabela6873 -- tratores

In [71]:
caminho_arquivo_entrada = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873.csv'


In [72]:
def extrair_transformar_carregar_tabela_tratores(caminho_arquivo_entrada):
    print("carregando csv bruto...")
    dataframe_bruto = pd.read_csv(
        caminho_arquivo_entrada, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    dataframe_bruto.columns = ['codigo_municipio', 'nome_localidade', 'condicao_produtor', 'codigo_filtro', 'grupos_atividade', 'valor_registrado']
    dataframe_bruto = dataframe_bruto.dropna(subset=['codigo_municipio'])
    
    # filtrar apenas linhas que possuem códigos ibge numéricos
    dataframe_bruto['is_ibge_numerico'] = dataframe_bruto['codigo_municipio'].astype(str).str.isnumeric()
    linhas_validas = dataframe_bruto[dataframe_bruto['is_ibge_numerico']].copy()
    
    # dividir o conjunto de dados ao meio (primeira metade: estabelecimentos, segunda metade: tratores)
    numero_de_registros = len(linhas_validas) // 2
    
    dataframe_estabelecimentos_dividido = linhas_validas.iloc[:numero_de_registros].copy()
    dataframe_maquinas_dividido = linhas_validas.iloc[numero_de_registros:].copy()
    
    print("processando valores numéricos...")
    dataframe_estabelecimentos_dividido['numero_de_estabelecimentos_agricolas'] = pd.to_numeric(dataframe_estabelecimentos_dividido['valor_registrado'], errors='coerce')
    dataframe_maquinas_dividido['numero_de_tratores_em_uso'] = pd.to_numeric(dataframe_maquinas_dividido['valor_registrado'], errors='coerce')
    
    # nós devemos fazer o merge usando todas as chaves compostas identificadoras para garantir uma correspondência 1:1
    chaves_mesclagem = ['codigo_municipio', 'nome_localidade', 'condicao_produtor', 'grupos_atividade', 'codigo_filtro']
    dataframe_maquinas_enxuto = dataframe_maquinas_dividido[chaves_mesclagem + ['numero_de_tratores_em_uso']]
    
    print("mesclando blocos de variáveis...")
    dataframe_final = pd.merge(
        dataframe_estabelecimentos_dividido,
        dataframe_maquinas_enxuto,
        on=chaves_mesclagem,
        how='left'
    )
    
    # extrair sigla_estado e limpar município do nome da nome_localidade
    dataframe_final['sigla_estado'] = dataframe_final['nome_localidade'].str.extract(r'\((.*?)\)$')[0]
    dataframe_final['nome_municipio'] = dataframe_final['nome_localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # calcular nível geo (ex: 1=brasil, 2=estado, 7=cidade)
    dataframe_final['nivel_geografico'] = dataframe_final['codigo_municipio'].astype(str).str.len()
    
    # remover colunas identificadoras desnecessárias resultantes do merge, e filtrar suas exclusões solicitadas
    colunas_para_remover = [
        'condicao_produtor', 'grupos_atividade', 'nivel_geografico', 
        'codigo_filtro', 'valor_registrado', 'is_ibge_numerico', 'nome_localidade'
    ]
    dataframe_final = dataframe_final.drop(columns=[c for c in colunas_para_remover if c in dataframe_final.columns], errors='ignore')
    
    # reordenar as métricas analíticas restantes de forma organizada
    colunas_esperadas = ['codigo_municipio', 'sigla_estado', 'nome_municipio', 'numero_de_estabelecimentos_agricolas', 'numero_de_tratores_em_uso']
    dataframe_final = dataframe_final[[c for c in colunas_esperadas if c in dataframe_final.columns]]
    
    # finalizar a aplicação do tipo inteiro correto para codigo_municipio
    dataframe_final['codigo_municipio'] = dataframe_final['codigo_municipio'].astype(int)
    
    return dataframe_final


In [73]:
dataframe_tratores_final = extrair_transformar_carregar_tabela_tratores(caminho_arquivo_entrada)
print(f"\netl concluído. formato final: {dataframe_tratores_final.shape}")
display(dataframe_tratores_final.head())


loading raw csv...
processing numeric values...
merging variable blocks...

etl completed. final shape: (5569, 5)


,municipality_code,state_abbreviation,municipality_name,number_of_agricultural_establishments,number_of_tractors_in_use
0,1,NaN,Brasil,734280.0,1229907.0
1,1,NaN,Norte,35092.0,58436.0
2,2,NaN,Nordeste,53284.0,83866.0
3,3,NaN,Sudeste,208791.0,373952.0
4,4,NaN,Sul,347476.0,517042.0


In [74]:
# salvar na pasta de saída
caminho_arquivo_csv_saida = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv'
print(f"\nsalvando em {caminho_arquivo_csv_saida}...")
dataframe_tratores_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding='utf-8')
print("concluído!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv...
done!


### 2.3 tabela6873 -- área em hectares

In [75]:
caminho_arquivo_entrada = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773.csv'


In [76]:
def extrair_transformar_carregar_tabela_estabelecimentos(caminho_arquivo_entrada):
    print("carregando csv bruto...")
    # carregar o arquivo bruto, ignorando as 4 primeiras linhas de metadados
    dataframe_bruto = pd.read_csv(
        caminho_arquivo_entrada, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    # renomear colunas para nomes internos padronizados
    dataframe_bruto.columns = ['codigo_municipio', 'nome_localidade', 'finalidade_uso_terra', 'faixa_renda', 'codigo_filtro', 'status_associacao', 'valor_registrado']
    # o conjunto de dados possui variáveis empilhadas: a primeira metade é 'número de estabelecimentos', a segunda metade é 'área em hectares'
    # limpar as notas de rodapé vazias
    dataframe_bruto = dataframe_bruto.dropna(subset=['codigo_municipio'])
    
    # filtrar apenas linhas representando códigos ibge válidos
    dataframe_bruto['is_ibge_numerico'] = dataframe_bruto['codigo_municipio'].astype(str).str.isnumeric()
    linhas_validas = dataframe_bruto[dataframe_bruto['is_ibge_numerico']].copy()
    
    # nós dividimos as linhas válidas pela metade: primeira metade = estabelecimentos, segunda metade = área
    numero_de_registros = len(linhas_validas) // 2
    
    dataframe_estabelecimentos_dividido = linhas_validas.iloc[:numero_de_registros].copy()
    dataframe_area_dividido = linhas_validas.iloc[numero_de_registros:].copy()
    
    print("processando colunas...")
    dataframe_estabelecimentos_dividido['numero_de_estabelecimentos_agricolas'] = pd.to_numeric(dataframe_estabelecimentos_dividido['valor_registrado'], errors='coerce')
    dataframe_area_dividido['area_total_em_hectares'] = pd.to_numeric(dataframe_area_dividido['valor_registrado'], errors='coerce')
    
    # extrair sigla_estado e limpar município de 'nome_localidade'
    dataframe_estabelecimentos_dividido['sigla_estado'] = dataframe_estabelecimentos_dividido['nome_localidade'].str.extract(r'\((.*?)\)$')[0]
    dataframe_estabelecimentos_dividido['nome_municipio'] = dataframe_estabelecimentos_dividido['nome_localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # calcular nível geo para possível filtragem hierárquica (1 = brasil, 7 = cidade)
    dataframe_estabelecimentos_dividido['nivel_geografico'] = dataframe_estabelecimentos_dividido['codigo_municipio'].astype(str).str.len()
    
    # remover colunas intermediárias que agora são redundantes
    dataframe_estabelecimentos_dividido = dataframe_estabelecimentos_dividido.drop(columns=['nome_localidade', 'valor_registrado', 'is_ibge_numerico'])
    dataframe_area_dividido = dataframe_area_dividido[['codigo_municipio', 'area_total_em_hectares']]
    
    print("mesclando blocos de variáveis...")
    # remontar as duas variáveis como colunas lado a lado
    dataframe_final = pd.merge(
        dataframe_estabelecimentos_dividido,
        dataframe_area_dividido,
        on='codigo_municipio',
        how='left'
    )
    
    # reordenar o dataframe final de forma organizada
    dataframe_final = dataframe_final[['codigo_municipio', 'nivel_geografico', 'sigla_estado', 'nome_municipio', 'finalidade_uso_terra', 'faixa_renda', 'status_associacao', 'numero_de_estabelecimentos_agricolas', 'area_total_em_hectares']]
    
    # ajustar tipos
    dataframe_final['codigo_municipio'] = dataframe_final['codigo_municipio'].astype(int)
    
    return dataframe_final


In [77]:
dataframe_estabelecimentos_final = extrair_transformar_carregar_tabela_estabelecimentos(caminho_arquivo_entrada)
print(f"etl concluído. formato final: {dataframe_estabelecimentos_final.shape}")
display(dataframe_estabelecimentos_final.head(30))


loading raw csv...
processing columns...
merging variable blocks...
etl completed. final shape: (5564, 9)


,municipality_code,geographic_level,state_abbreviation,municipality_name,land_use_purpose,income_bracket,association_status,number_of_agricultural_establishments,total_area_in_hectares
0,1,1,NaN,Brasil,Total,Total,Total,5073324,351289816.0
1,1100015,7,RO,Alta Floresta D'Oeste,Total,Total,Total,2886,372746.0
2,1100023,7,RO,Ariquemes,Total,Total,Total,2928,334256.0
3,1100031,7,RO,Cabixi,Total,Total,Total,1075,113085.0
4,1100049,7,RO,Cacoal,Total,Total,Total,3814,221390.0
5,1100056,7,RO,Cerejeiras,Total,Total,Total,719,126686.0
6,1100064,7,RO,Colorado do Oeste,Total,Total,Total,1674,139796.0
7,1100072,7,RO,Corumbiara,Total,Total,Total,1489,273086.0
8,1100080,7,RO,Costa Marques,Total,Total,Total,1500,220177.0
9,1100098,7,RO,Espigão D'Oeste,Total,Total,Total,2000,247331.0


In [78]:
# formatos de saída
caminho_arquivo_csv_saida = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv'
print(f"\nsalvando em {caminho_arquivo_csv_saida}...")
dataframe_estabelecimentos_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding='utf-8')
print("concluído!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv...
done!


### 2.4 tabela6873 -- produtos / área colhida

In [79]:
caminho_arquivo_entrada = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612.csv'


In [80]:
# gerar os nomes corretos das colunas
anos = range(1974, 2025)
produtos = [
    'Algodão herbáceo (em caroço)',
    'Cana-de-açúcar',
    'Milho (em grão)',
    'Soja (em grão)'
]
columns = ['codigo_municipio', 'nome_localidade']
for indice_ano in anos:
    for indice_produto in produtos:
        columns.append(f"{indice_ano}_{indice_produto}")
print(f"esperado {len(columns)} columns.")


expected 206 columns.


In [81]:
# tentar ler a primeira linha para ver quantas colunas realmente possui
with open(caminho_arquivo_entrada, 'r', encoding='utf-8') as f:
    for _ in range(5):
        next(f)
    primeira_linha_dados = f.readline().strip()
numero_de_colunas_dados = len(primeira_linha_dados.split(';'))
print(f"linhas de dados parecem ter {numero_de_colunas_dados} columns.")


data rows seem to have 206 columns.


In [82]:
# se houver um ponto e vírgula final extra, podemos ler uma coluna vazia extra
colunas_para_usar = list(range(len(columns)))


In [83]:
# motor c rápido
dataframe = pd.read_csv(
    caminho_arquivo_entrada, 
    sep=';', 
    skiprows=5,
    names=columns,
    usecols=colunas_para_usar,
    na_values=['-', 'X', '..', '...', ' '],
    decimal=',',
    engine='c', # usar motor c para velocidade
    low_memory=False
)
print(f"formato do csv carregado: {dataframe.shape}")


loaded csv shape: (11167, 206)


In [84]:
# remover linhas que são apenas notas no final do csv (codigo_municipio deve ser numérico)
dataframe = dataframe[pd.to_numeric(dataframe['codigo_municipio'], errors='coerce').notnull()]
dataframe['codigo_municipio'] = dataframe['codigo_municipio'].astype(int)


In [85]:
# identificar níveis (brasil=1, região=1, estado=2, cidade=7 geralmente)
dataframe['codigo_municipio_string'] = dataframe['codigo_municipio'].astype(str)
dataframe['nivel_geografico'] = dataframe['codigo_municipio_string'].str.len()


In [86]:
# fazer o melt do dataframe para formato longo: codigo_municipio, nome_localidade, nivel_geografico, ano_referencia, produto_agricola, area_colhida
variaveis_identificadoras = ['codigo_municipio', 'nome_localidade', 'nivel_geografico']
variaveis_valor = [col for col in columns if col not in ['codigo_municipio', 'nome_localidade']]
print("aplicando melt no dataframe...")
dataframe_derretido = dataframe.melt(id_vars=variaveis_identificadoras, value_vars=variaveis_valor, var_name='ano_e_produto', value_name='area_colhida_em_hectares')
print("extracting indice_ano and product...")
dataframe_derretido[['ano_referencia', 'produto_agricola']] = dataframe_derretido['ano_e_produto'].str.extract(r'^(\d{4})_(.*)$')
dataframe_derretido.drop(columns=['ano_e_produto'], inplace=True)
dataframe_derretido['ano_referencia'] = dataframe_derretido['ano_referencia'].astype(int)
dataframe_derretido['area_colhida_em_hectares'] = pd.to_numeric(dataframe_derretido['area_colhida_em_hectares'], errors='coerce')
dataframe_derretido.dropna(subset=['area_colhida_em_hectares'], inplace=True)


melting dataframe...
extracting year_index and product...


In [87]:
# limpar nome da nome_localidade
dataframe_derretido['sigla_estado'] = dataframe_derretido['nome_localidade'].str.extract(r'\((.*?)\)$')[0]
dataframe_derretido['nome_municipio'] = dataframe_derretido['nome_localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)


In [88]:
# reordenar colunas
dataframe_final = dataframe_derretido[['codigo_municipio', 'nivel_geografico', 'sigla_estado', 'nome_municipio', 'ano_referencia', 'produto_agricola', 'area_colhida_em_hectares']]
print(f"etl concluído. formato final: {dataframe_final.shape}")
display(dataframe_final.head(30))


etl completed. final shape: (1065021, 7)


,municipality_code,geographic_level,state_abbreviation,municipality_name,reference_year,agricultural_product,harvested_area_in_hectares
0,1,1,NaN,Brasil,1974,Algodão herbáceo (em caroço),1726036.0
1,1,1,NaN,Norte,1974,Algodão herbáceo (em caroço),256.0
2,2,1,NaN,Nordeste,1974,Algodão herbáceo (em caroço),809101.0
3,3,1,NaN,Sudeste,1974,Algodão herbáceo (em caroço),496122.0
4,4,1,NaN,Sul,1974,Algodão herbáceo (em caroço),310000.0
5,5,1,NaN,Centro-Oeste,1974,Algodão herbáceo (em caroço),110557.0
22,1100205,7,RO,Porto Velho,1974,Algodão herbáceo (em caroço),200.0
168,1500909,7,PA,Augusto Corrêa,1974,Algodão herbáceo (em caroço),9.0
180,1501709,7,PA,Bragança,1974,Algodão herbáceo (em caroço),27.0
190,1502202,7,PA,Capanema,1974,Algodão herbáceo (em caroço),12.0


In [89]:
# salvar em um novo csv
caminho_arquivo_csv_saida = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv'
print(f"salvando em {caminho_arquivo_csv_saida}...")
dataframe_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding='utf-8')
print("concluído!")


saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv...
done!


### 2.4 ibge população

In [90]:
caminho_arquivo_entrada = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio.csv'


In [91]:
def extrair_transformar_carregar_populacao(caminho_arquivo_entrada):
    print("carregando dados populacionais...")
    dataframe = pd.read_csv(caminho_arquivo_entrada, sep=',')
    
    # rename columns to english immediately to avoid keyerrors
    rename_map = {
        'ano': 'ano_referencia', 
        'id_municipio': 'codigo_municipio', 
        'sigla_uf': 'sigla_estado', 
        'populacao': 'populacao_total'
    }
    dataframe = dataframe.rename(columns=rename_map)
    
    print(f"formato original: {dataframe.shape}")
    print("anos disponíveis:", sorted(dataframe['ano_referencia'].unique()))
    
    # limpando os tipos de dados
    dataframe = dataframe.dropna(subset=['codigo_municipio'])
    dataframe['codigo_municipio'] = dataframe['codigo_municipio'].astype(int)
    
    return dataframe

In [92]:
dataframe_populacao_final = extrair_transformar_carregar_populacao(caminho_arquivo_entrada)
print(f"\netl concluído. formato final: {dataframe_populacao_final.shape}")
display(dataframe_populacao_final.head(10))


carregando dados populacionais...
formato original: (191099, 4)
anos disponíveis: [np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

etl completed. final shape: (191098, 4)


,reference_year,state_abbreviation,municipality_code,total_population
0,1991,RO,1100015,31981.0
1,1991,RO,1100023,83684.0
2,1991,RO,1100031,8173.0
3,1991,RO,1100049,79036.0
4,1991,RO,1100056,21607.0
5,1991,RO,1100064,38994.0
6,1991,RO,1100080,10375.0
7,1991,RO,1100098,23157.0
8,1991,RO,1100106,32583.0
9,1991,RO,1100114,63535.0


In [93]:
# salvar saída
caminho_arquivo_csv_saida = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv'
print(f"\nsalvando em {caminho_arquivo_csv_saida}...")
dataframe_populacao_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding='utf-8')
print("concluído!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv...
done!


## 3. cruzando os dados

In [94]:
# carregar todos os conjuntos de dados limpos que construímos!
print("loading individual datasets...")
dataframe_populacao = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv')
dataframe_educacao = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv')
dataframe_agricultura = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv')
dataframe_estabelecimentos = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv')
dataframe_tratores = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv')
print("applying cross-reference logic via 'codigo_municipio'...")


loading individual datasets...


applying cross-reference logic via 'municipality_code'...


In [95]:
# 1. tabela dataframe_base: usamos a métrica de população de 2024 como âncora já que valida todos os 5.570 códigos ibge ativos reais
dataframe_base = dataframe_populacao[dataframe_populacao['ano_referencia'] == 2024][['codigo_municipio', 'sigla_estado', 'populacao_total']].copy()
dataframe_base = dataframe_base.rename(columns={'populacao_total': 'populacao_em_2024'})


In [96]:
# 2. adicionar dados de educação
# o conjunto de dados possui múltiplas linhas por school_location/dependencia_administrativa, então nós agrupamos seguramente apenas por código ibge para obter o total de escolas dentro da fronteira municipal
aggregated_education_data = dataframe_educacao.groupby('codigo_municipio')['matriculas_ensino_medio_rural'].sum().reset_index()
dataframe_base = pd.merge(dataframe_base, aggregated_education_data, on='codigo_municipio', how='left')


In [97]:
# 3. adicionar área agrícola (tabela 1612 - histórico)
# vamos filtrar pelo ano_referencia estatístico mais recente registrado (2024 ou 2023 dependendo da cultura)
ano_maximo = dataframe_agricultura['ano_referencia'].max()
dataframe_agricultura_recente = dataframe_agricultura[dataframe_agricultura['ano_referencia'] == ano_maximo]


In [98]:
# pivotar os produtos agrícolas dinamicamente para termos uma linha por município, e produtos distintos agindo como colunas simples horizontais
dataframe_agricultura_pivo = dataframe_agricultura_recente.pivot_table(
    index='codigo_municipio', 
    columns='produto_agricola', 
    values='area_colhida_em_hectares', 
    aggfunc='sum'
).reset_index()


In [99]:
# renomear strings do cabeçalho para snake_case evitando espaços
dataframe_agricultura_pivo.columns = ['codigo_municipio'] + [f"area_{c.split(' (')[0].replace('-', '_').replace(' ', '_').lower()}_{ano_maximo}" for c in dataframe_agricultura_pivo.columns if c != 'codigo_municipio']
dataframe_base = pd.merge(dataframe_base, dataframe_agricultura_pivo, on='codigo_municipio', how='left')


In [100]:
# 4. adicionar área de estabelecimentos agropecuários (tabela 6773)
# para evitar a dupla correspondência dos cabeçalhos de estado/região, vamos filtrar rigorosamente o nivel_geografico apenas para '7' (nível cidade)
if 'nivel_geografico' in dataframe_estabelecimentos.columns:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos[dataframe_estabelecimentos['nivel_geografico'] == 7].drop_duplicates(subset=['codigo_municipio'])
else:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos.drop_duplicates(subset=['codigo_municipio'])
dataframe_estabelecimentos_municipio = dataframe_estabelecimentos_municipio[['codigo_municipio', 'numero_de_estabelecimentos_agricolas', 'area_total_em_hectares']]
dataframe_estabelecimentos_municipio.columns = ['codigo_municipio', 'total_estabelecimentos_agricolas', 'area_total_agricola_em_hectares']
dataframe_base = pd.merge(dataframe_base, dataframe_estabelecimentos_municipio, on='codigo_municipio', how='left')


In [101]:
# 5. adicionar dados de uso de tratores (tabela 6873)
dataframe_tratores_municipio = dataframe_tratores.drop_duplicates(subset=['codigo_municipio'])
dataframe_tratores_municipio = dataframe_tratores_municipio[['codigo_municipio', 'numero_de_tratores_em_uso']]
dataframe_base = pd.merge(dataframe_base, dataframe_tratores_municipio, on='codigo_municipio', how='left')
print(f"\nformato unificado final de dados (1 linha por município): {dataframe_base.shape}")
display(dataframe_base.head(30))



final unified data shape (1 row per municipality): (5570, 11)


,municipality_code,state_abbreviation,population_in_2024,rural_high_school_enrollments,area_algodão_herbáceo_2024,area_cana_de_açúcar_2024,area_milho_2024,area_soja_2024,total_agricultural_establishments,total_agricultural_area_in_hectares,number_of_tractors_in_use
0,1100015,RO,22853.0,72,NaN,84.0,9664.0,11758.0,2886.0,372746.0,517.0
1,1100023,RO,108573.0,703,NaN,730.0,37593.0,109520.0,2928.0,334256.0,501.0
2,1100031,RO,5690.0,33,NaN,92.0,104069.0,224895.0,1075.0,113085.0,247.0
3,1100049,RO,97637.0,189,NaN,673.0,16302.0,43797.0,3814.0,221390.0,465.0
4,1100056,RO,16975.0,38,NaN,616.0,129356.0,265911.0,719.0,126686.0,303.0
5,1100064,RO,16588.0,575,NaN,59.0,8133.0,15653.0,1674.0,139796.0,248.0
6,1100072,RO,8001.0,96,NaN,209.0,161976.0,356007.0,1489.0,273086.0,306.0
7,1100080,RO,13522.0,175,NaN,105.0,19549.0,48038.0,1500.0,220177.0,269.0
8,1100098,RO,32717.0,39,NaN,NaN,6720.0,12593.0,2000.0,247331.0,334.0
9,1100106,RO,43553.0,120,NaN,25.0,10998.0,63630.0,602.0,70487.0,91.0


In [102]:
# salvar conjunto de dados master de referência
caminho_arquivo_csv_master_saida = '/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv'
print(f"\nsalvando em {caminho_arquivo_csv_master_saida} ...")
dataframe_base.to_csv(caminho_arquivo_csv_master_saida, index=False, encoding='utf-8')
print("concluído!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv ...
complete!


## 4. master granular (dataframe_base expandida)
ao invés de agrupar tudo em apenas 5.570 linhas, criamos uma visão detalhada para atingir +1.5m de registros.

In [103]:
# gerar dataset massivo
print("iniciando mescla granular com matriculas rurais...")
dataframe_agricultura_expandido = dataframe_agricultura.copy()

iniciando mescla granular com matriculas rurais...


In [104]:
# nova coluna matriculas_ensino_medio_rural e sem school_location que era fixa
dados_dependencias_educacao = dataframe_educacao[['codigo_municipio', 'dependencia_administrativa', 'matriculas_ensino_medio_rural']].copy()

In [ ]:
# 1. cruzamento cartesiano (agricultura histórica x dependência escolar)
dataframe_grade_final = pd.merge(dataframe_agricultura_expandido, dados_dependencias_educacao, on='codigo_municipio', how='outer')

In [106]:
# 2. trazer a população por ano_referencia correspondente
dataframe_grade_final = pd.merge(dataframe_grade_final, dataframe_populacao[['codigo_municipio', 'ano_referencia', 'populacao_total']], on=['codigo_municipio', 'ano_referencia'], how='left')

In [107]:
# 3. trazer estatísticas estáticas (área e tratores)
if 'nivel_geografico' in dataframe_estabelecimentos.columns:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos[dataframe_estabelecimentos['nivel_geografico'] == 7].drop_duplicates(subset=['codigo_municipio'])
else:
    dataframe_estabelecimentos_municipio = dataframe_estabelecimentos.drop_duplicates(subset=['codigo_municipio'])
dataframe_estabelecimentos_municipio = dataframe_estabelecimentos_municipio[['codigo_municipio', 'numero_de_estabelecimentos_agricolas', 'area_total_em_hectares']]
dataframe_estabelecimentos_municipio.columns = ['codigo_municipio', 'total_estabelecimentos_agricolas', 'area_total_agricola_em_hectares']

dataframe_grade_final = pd.merge(dataframe_grade_final, dataframe_estabelecimentos_municipio, on='codigo_municipio', how='left')

dataframe_tratores_municipio = dataframe_tratores.drop_duplicates(subset=['codigo_municipio'])[['codigo_municipio', 'numero_de_tratores_em_uso']]
dataframe_grade_final = pd.merge(dataframe_grade_final, dataframe_tratores_municipio, on='codigo_municipio', how='left')

In [108]:
# limpar nas em produto_agricola
dataframe_grade_final['produto_agricola'] = dataframe_grade_final['produto_agricola'].fillna('n/a (apenas dados escolares/estáticos)')

print(f"granular shape: {dataframe_grade_final.shape}")

caminho_arquivo_csv_saida = '/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios.csv'
print(f"\nsalvando em {caminho_arquivo_csv_saida}...")
dataframe_grade_final.to_csv(caminho_arquivo_csv_saida, index=False, encoding='utf-8')
print("finalizado!")

granular shape: (4252902, 13)

salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios.csv...
finalizado!
